# 07 Equal factor for all blocks and a positive control (added after the protocol, review round 2)
Three sets of 25 runs each (5 folds x 5 seeds), with the same folds, seeds, initial weights and code as the main runs:
1. **Tr4 with k = 100** (plan A). With Q4, S4, Cl4 (k = 100) and Qp4 (k = 100, notebook 05), family F1 then has every block at the same factor.
2. **Tr8 with k = 100** (plan B). With Q8 and Qp8 at k = 100 (notebook 05 and main runs), family F4 at eight qubits then has every block at the same factor.
3. **Z4, a positive control** (plan A): the slot outputs zeros, so the decoder gets no information from the code. It shows whether this slot can reveal a block that is known to be worse.

Outputs go to `runs_rev2/` only; `runs/`, `runs_k100/` and `runs_min30/` are not touched. **T4 GPU only.** Two sessions can run at the same time: set `SESSION = 'B'` in the second one. Do not change `CFG`; cell 1 checks it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time
ROOT = '/content/drive/MyDrive/BUSI_QB'
CODE = os.path.join(ROOT, 'code'); PREP = os.path.join(ROOT, 'prepared')
RUNS_R = os.path.join(ROOT, 'runs_rev2'); os.makedirs(RUNS_R, exist_ok=True)
sys.path.insert(0, os.path.join(CODE, 'src'))
SESSION = 'A'                                     # 'B' in a second session
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
import qsim, qbus, runner
print(qbus.env_info())
assert 'T4' in str(qbus.env_info().get('gpu')), 'select a T4 GPU runtime'
# protocol settings must be unchanged (a debugging override caused the 23 short runs earlier)
assert qbus.CFG['min_epochs'] == 30 and qbus.CFG['max_epochs'] == 200 and qbus.CFG['es_patience'] == 20, qbus.CFG
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)

# zero block 'z': the code is replaced by zeros inside the network (no weights; not named 'block')
_orig_make_block = qbus.make_block
def _make_block(kind, n, k):
    if kind == 'z':
        return keras.layers.Lambda(lambda t: tf.zeros_like(t), name='zero_block')
    return _orig_make_block(kind, n, k)
qbus.make_block = _make_block
if 'z' not in qbus.BLOCKS:
    qbus.BLOCKS = qbus.BLOCKS + ('z',)

# smoke check of the zero block (no training)
m = qbus.build_model('z4', k=1)
print('init diff', qbus.apply_init(m, qbus.reference_init(4, 0)))
c = keras.Model(m.input, m.get_layer('zero_block').output)(np.random.rand(2, 256, 256, 1).astype('float32'))
assert float(tf.reduce_max(tf.abs(c))) == 0.0
del m, c; runner.free_memory()

todo  = [dict(plan='A', model='tr4', fold=f, seed=s, frac=1.0, k=100) for s in runner.SEEDS for f in range(5)]
todo += [dict(plan='A', model='z4',  fold=f, seed=s, frac=1.0, k=1)   for s in runner.SEEDS for f in range(5)]
todo += [dict(plan='B', model='tr8', fold=f, seed=s, frac=1.0, k=100) for s in runner.SEEDS for f in range(5)]
todo = [r for trio in zip(todo[:25], todo[25:50], todo[50:]) for r in trio]     # alternate the three sets
done = glob.glob(os.path.join(RUNS_R, '*', '*', 'summary.json'))
print(len(todo), 'runs in total,', len(done), 'done')

In [ ]:
# 2. RUNS ----------------------------------------------------------------------------------------------
for r in runner.ordered(todo, SESSION):
    root = os.path.join(RUNS_R, r['plan']); os.makedirs(root, exist_ok=True)
    rid = runner.run_id(r)
    if os.path.exists(os.path.join(root, rid, 'summary.json')):
        continue
    if runner.claim(root, rid, SESSION):
        print(time.strftime('%m-%d %H:%M'), 'start', rid, 'k =', r['k'])
        s = runner.one_run(r, PREP, root, session=SESSION)
        assert s['epochs'] >= 30, ('stopped before 30 epochs', rid)
        print('   Dice', round(s['test_dice'], 4), 'k', s['k'], 'epochs', s.get('epochs'), 'block change',
              round(s.get('change_block', float('nan')), 4), 'min', round(s['minutes'], 1))

In [ ]:
# 3. COLLECT (when all 75 runs are done) -----------------------------------------------------------------
N = 75
dirs = sorted(glob.glob(os.path.join(RUNS_R, '*', '*')))
res = []
for d in dirs:
    s = os.path.join(d, 'summary.json')
    if not os.path.exists(s):
        continue
    out = dict(summary=json.load(open(s)))
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement'):
        f = os.path.join(d, nm + '.csv')
        if os.path.exists(f):
            t = pd.read_csv(f); t.insert(0, 'run_id', out['summary']['run_id']); out[nm] = t
    res.append(out)
print(f'done {len(res)} / {N}, running {len(glob.glob(os.path.join(RUNS_R, "*", "*.lock")))}')
if len(res) == N:
    OUT = os.path.join(ROOT, 'analysis', 'rev2'); os.makedirs(OUT, exist_ok=True)
    summ = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in r['summary'].items()} for r in res])
    assert (summ.epochs >= 30).all(), summ.loc[summ.epochs < 30, 'run_id'].tolist()
    summ.to_csv(os.path.join(OUT, 'summaries.csv'), index=False)
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement'):
        parts = [r[nm] for r in res if nm in r]
        if parts:
            t = pd.concat(parts, ignore_index=True)
            for c in t.columns:
                if t[c].dtype == float:
                    t[c] = t[c].round(5)
            t.to_csv(os.path.join(OUT, nm + '.csv.gz'), index=False, compression='gzip')
            print(nm, len(t), 'rows')
    print(summ.groupby('model').agg(runs=('run_id', 'size'), dice=('test_dice', 'mean'), epochs_min=('epochs', 'min'),
                                    change=('change_block', 'median')).round(4))
    print('Files written to', OUT)